# BoldSearch: MP4 → V1 → Milvus → web app → Cloudflare

Chạy **Run All** trong Kaggle sau khi bật GPU, bật Internet (nếu không attach model FG-CLIP2 offline), attach dataset MP4 và attach AutoShot source + checkpoint. Notebook clone app `main` riêng với runtime branch riêng; không sửa source của app clone.

Kaggle Secrets bắt buộc: `ZILLIZ_URI`, `ZILLIZ_TOKEN`. `HF_TOKEN` là tùy chọn khi dùng Hugging Face online.

In [ ]:
import json
import os
import re
import shutil
import signal
import subprocess
import sys
import time
from pathlib import Path

APP_REPO_URL = 'https://github.com/ToiLaKiet/BoldSearch.git'
APP_REPO_REF = 'main'
RUNTIME_REPO_URL = APP_REPO_URL
RUNTIME_REPO_REF = "feat/kaggle-mp4-run-all"

APP_REPO_ROOT = Path('/kaggle/working/BoldSearch')
RUNTIME_REPO_ROOT = Path('/kaggle/working/boldsearch-runtime-code')
RUNTIME_ROOT = Path('/kaggle/working/boldsearch-runtime')
PIPELINE_ROOT = RUNTIME_REPO_ROOT / 'pipelines/aic_video_pipeline_v1'
PIPELINE_DATA_ROOT = RUNTIME_ROOT / 'pipeline-data'
PUBLIC_ROOT = RUNTIME_ROOT / 'public'
FRONTEND_DIST = PUBLIC_ROOT / 'frontend-dist'

# Dùng collection mới, visual-only. Không ghi vào collection legacy.
CORPUS_VERSION = 'aic2026-l21-v1'
COLLECTION_NAME = 'BoldSearchV1_AIC2026_L21'
PIPELINE_PROFILE = 'default.yaml'  # legacy_compatible.yaml chỉ dùng khi golden-compare collection cũ
SELECTED_VIDEO_IDS = None  # None = mọi MP4 được attach; hoặc ['L21_V001']
FORCE_FRESH_PIPELINE = False

# Đặt path tuyệt đối nếu auto-discovery không tìm được AutoShot/model offline.
AUTOSHOT_ROOT_OVERRIDE = None
AUTOSHOT_CHECKPOINT_OVERRIDE = None
FGCLIP_MODEL_PATH_OVERRIDE = None  # None = tải qihoo360/fg-clip2-large qua Internet

BACKEND_PORT = 8000
GATEWAY_PORT = 7860
SEARCH_TOP_K = 50
THUMBNAIL_WIDTH = 960
WEBP_QUALITY = 82

if not Path('/kaggle/input').is_dir() or not Path('/kaggle/working').is_dir():
    raise RuntimeError('Notebook này phải chạy trên Kaggle.')
if not shutil.which('git') or not shutil.which('ffmpeg') or not shutil.which('npm'):
    raise RuntimeError('Kaggle image thiếu git, ffmpeg hoặc npm.')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Hãy bật GPU trong Kaggle Notebook Settings rồi restart session.')

for path in (RUNTIME_ROOT, PIPELINE_DATA_ROOT, PUBLIC_ROOT):
    path.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0))
print('Runtime branch:', RUNTIME_REPO_REF)


In [ ]:
def clone_or_verify(url: str, ref: str, root: Path, required: list[Path]) -> str:
    if root.exists():
        if not (root / '.git').is_dir():
            raise RuntimeError(f'{root} exists but is not a Git repository.')
        origin = subprocess.check_output(
            ['git', '-C', str(root), 'remote', 'get-url', 'origin'], text=True
        ).strip()
        if origin.rstrip('/') != url.rstrip('/'):
            raise RuntimeError(f'Unexpected origin for {root}: {origin}')
        dirty = subprocess.check_output(
            ['git', '-C', str(root), 'status', '--porcelain', '--untracked-files=no'], text=True
        ).strip()
        if dirty:
            raise RuntimeError(f'Clone has tracked changes; use a fresh Kaggle session:\n{dirty}')
        subprocess.run(['git', '-C', str(root), 'fetch', '--depth', '1', 'origin', ref], check=True)
        subprocess.run(['git', '-C', str(root), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', ref, url, str(root)], check=True)
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise RuntimeError(f'Clone {root} is missing required files:\n' + '\n'.join(missing))
    return subprocess.check_output(['git', '-C', str(root), 'rev-parse', 'HEAD'], text=True).strip()

app_commit = clone_or_verify(APP_REPO_URL, APP_REPO_REF, APP_REPO_ROOT, [
    APP_REPO_ROOT / 'app/backend/pyproject.toml',
    APP_REPO_ROOT / 'app/backend/uv.lock',
    APP_REPO_ROOT / 'app/frontend/package-lock.json',
    APP_REPO_ROOT / 'app/frontend/src/App.jsx',
])
runtime_commit = clone_or_verify(RUNTIME_REPO_URL, RUNTIME_REPO_REF, RUNTIME_REPO_ROOT, [
    RUNTIME_REPO_ROOT / 'boldsearch_integration/cli.py',
    RUNTIME_REPO_ROOT / 'boldsearch_integration/fastapi_launcher.py',
    PIPELINE_ROOT / 'src/aic_video_pipeline_v1/orchestrator.py',
    PIPELINE_ROOT / 'configs/default.yaml',
])
BACKEND_ROOT = APP_REPO_ROOT / 'app/backend'
FRONTEND_ROOT = APP_REPO_ROOT / 'app/frontend'
PIPELINE_CONFIG = PIPELINE_ROOT / 'configs' / PIPELINE_PROFILE
if not PIPELINE_CONFIG.is_file():
    raise RuntimeError(f'Pipeline config does not exist: {PIPELINE_CONFIG}')
print('App commit:', app_commit)
print('Runtime commit:', runtime_commit)


In [ ]:
def one_path(label: str, candidates: list[Path]) -> Path:
    unique = sorted({path.resolve() for path in candidates})
    if len(unique) != 1:
        rendered = '\n'.join(map(str, unique[:20]))
        raise RuntimeError(f'Need exactly one {label}; found {len(unique)}:\n{rendered}')
    return unique[0]

video_re = re.compile(r'^L(?P<level>\d{2})_V(?P<number>\d{2,3})\.mp4$')
group_re = re.compile(r'^Videos_L(?P<level>\d{2})_[a-z0-9]+$')
all_videos = sorted(Path('/kaggle/input').glob('**/Videos_L??_*/video/L??_V???.mp4'))
if not all_videos:
    raise RuntimeError('Attach an AIC MP4 dataset containing Videos_Lxx_*/video/Lxx_Vnnn.mp4.')
VIDEO_PATHS = {}
for video in all_videos:
    match = video_re.fullmatch(video.name)
    group = group_re.fullmatch(video.parent.parent.name)
    if not match or not group or match.group('level') != group.group('level'):
        raise RuntimeError(f'Invalid AIC MP4 layout: {video}')
    if video.stem in VIDEO_PATHS:
        raise RuntimeError(f'Duplicate video ID across mounted datasets: {video.stem}')
    VIDEO_PATHS[video.stem] = video.resolve()
if SELECTED_VIDEO_IDS is None:
    PROCESS_VIDEOS = [VIDEO_PATHS[key] for key in sorted(VIDEO_PATHS)]
else:
    missing = sorted(set(SELECTED_VIDEO_IDS) - set(VIDEO_PATHS))
    if missing:
        raise RuntimeError('Selected videos are not mounted: ' + ', '.join(missing))
    PROCESS_VIDEOS = [VIDEO_PATHS[key] for key in SELECTED_VIDEO_IDS]

autoshot_root = (Path(AUTOSHOT_ROOT_OVERRIDE).resolve() if AUTOSHOT_ROOT_OVERRIDE else None)
if autoshot_root is None:
    autoshot_root = one_path(
        'AutoShot source directory',
        [path.parent for path in Path('/kaggle/input').glob('**/supernet_flattransf_3_8_8_8_13_12_0_16_60.py')],
    )
autoshot_checkpoint = (Path(AUTOSHOT_CHECKPOINT_OVERRIDE).resolve() if AUTOSHOT_CHECKPOINT_OVERRIDE else None)
if autoshot_checkpoint is None:
    autoshot_checkpoint = one_path(
        'AutoShot checkpoint', list(Path('/kaggle/input').glob('**/ckpt_*.pth'))
    )
if not autoshot_root.is_dir() or not autoshot_checkpoint.is_file():
    raise RuntimeError('AutoShot root/checkpoint is invalid.')
FGCLIP_MODEL_PATH = (Path(FGCLIP_MODEL_PATH_OVERRIDE).resolve() if FGCLIP_MODEL_PATH_OVERRIDE else None)
if FGCLIP_MODEL_PATH is not None and not FGCLIP_MODEL_PATH.is_dir():
    raise RuntimeError(f'FG-CLIP model path is not a directory: {FGCLIP_MODEL_PATH}')
print('Videos:', ', '.join(path.stem for path in PROCESS_VIDEOS))
print('AutoShot root:', autoshot_root)
print('AutoShot checkpoint:', autoshot_checkpoint)
print('FG-CLIP source:', FGCLIP_MODEL_PATH or 'Hugging Face download')


In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
except Exception:
    _secrets = None

def read_secret(name: str, required: bool = True) -> str:
    value = str(os.environ.get(name, '')).strip()
    if not value and _secrets is not None:
        try:
            value = str(_secrets.get_secret(name) or '').strip()
        except Exception:
            value = ''
    if required and not value:
        raise RuntimeError(f'Missing Kaggle Secret: {name}')
    if '\n' in value or '\r' in value:
        raise RuntimeError(f'Secret {name} must not contain a newline.')
    return value

ZILLIZ_URI = read_secret('ZILLIZ_URI')
ZILLIZ_TOKEN = read_secret('ZILLIZ_TOKEN')
HF_TOKEN = read_secret('HF_TOKEN', required=False)
if FGCLIP_MODEL_PATH is None and not HF_TOKEN:
    print('HF_TOKEN is empty: public model download is attempted; attach FG-CLIP offline model if it is gated or Internet is disabled.')

if shutil.which('uv') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv'], check=True)
subprocess.run(['uv', 'sync', '--frozen', '--no-dev'], cwd=BACKEND_ROOT, check=True)
BACKEND_PYTHON = BACKEND_ROOT / '.venv/bin/python'
if not BACKEND_PYTHON.is_file():
    raise RuntimeError(f'uv did not create backend Python: {BACKEND_PYTHON}')
subprocess.run([
    'uv', 'pip', 'install', '--python', str(BACKEND_PYTHON), '--quiet', '-e', str(PIPELINE_ROOT)
], check=True)
subprocess.run(['npm', 'ci', '--ignore-scripts', '--no-audit', '--no-fund'], cwd=FRONTEND_ROOT, check=True)
build_env = os.environ.copy()
build_env.update({
    'BOLDSEARCH_FRONTEND_ROOT': str(FRONTEND_ROOT),
    'BOLDSEARCH_FRONTEND_DIST': str(FRONTEND_DIST),
})
subprocess.run([
    'npm', 'exec', '--', 'vite', 'build', '--config',
    str(RUNTIME_REPO_ROOT / 'boldsearch_integration/vite.runtime.mjs'),
], cwd=FRONTEND_ROOT, env=build_env, check=True)
if not (FRONTEND_DIST / 'index.html').is_file():
    raise RuntimeError('Vite did not produce frontend index.html.')
built_js = '\n'.join(path.read_text(encoding='utf-8', errors='replace') for path in FRONTEND_DIST.rglob('*.js'))
if 'http://0.0.0.0:8000/api' in built_js:
    raise RuntimeError('Frontend build still contains the old absolute API URL.')
print('Dependencies and frontend build are ready.')


In [ ]:
runtime_env = os.environ.copy()
runtime_env['PYTHONPATH'] = str(RUNTIME_REPO_ROOT)
if HF_TOKEN:
    runtime_env['HF_TOKEN'] = HF_TOKEN
command = [
    str(BACKEND_PYTHON), '-m', 'boldsearch_integration.cli', 'run',
    '--pipeline-root', str(PIPELINE_ROOT),
    '--config', str(PIPELINE_CONFIG),
    '--data-root', str(PIPELINE_DATA_ROOT),
    '--output-root', str(PUBLIC_ROOT),
    '--autoshot-root', str(autoshot_root),
    '--autoshot-checkpoint', str(autoshot_checkpoint),
    '--corpus-version', CORPUS_VERSION,
    '--thumbnail-width', str(THUMBNAIL_WIDTH),
    '--webp-quality', str(WEBP_QUALITY),
]
for video in PROCESS_VIDEOS:
    command.extend(['--video', str(video)])
if FGCLIP_MODEL_PATH is not None:
    command.extend(['--model-path', str(FGCLIP_MODEL_PATH)])
if FORCE_FRESH_PIPELINE:
    command.append('--fresh')
run_result = subprocess.run(command, env=runtime_env, check=True, text=True, capture_output=True)
print(run_result.stdout.strip())
active_path = PUBLIC_ROOT / 'active.json'
if not active_path.is_file():
    raise RuntimeError('Pipeline completed without an active release.')
active_release_id = json.loads(active_path.read_text(encoding='utf-8'))['release_id']
ACTIVE_RELEASE = PUBLIC_ROOT / 'releases' / active_release_id
manifest = json.loads((ACTIVE_RELEASE / 'corpus-manifest.json').read_text(encoding='utf-8'))
if manifest.get('corpus_version') != CORPUS_VERSION:
    raise RuntimeError('Published corpus version does not match this run.')
print(f'Published {manifest["row_count"]} keyframes in {ACTIVE_RELEASE}')


In [ ]:
bootstrap = [
    str(BACKEND_PYTHON), '-m', 'boldsearch_integration.cli', 'bootstrap',
    '--collection', COLLECTION_NAME, '--expected-vector-dim', '1024',
    '--uri', ZILLIZ_URI, '--token', ZILLIZ_TOKEN,
]
print(subprocess.check_output(bootstrap, env=runtime_env, text=True).strip())
ingest = [
    str(BACKEND_PYTHON), '-m', 'boldsearch_integration.cli', 'ingest',
    '--data-root', str(PIPELINE_DATA_ROOT),
    '--corpus-version', CORPUS_VERSION,
    '--collection', COLLECTION_NAME,
    '--uri', ZILLIZ_URI, '--token', ZILLIZ_TOKEN,
    '--batch-size', '128', '--retries', '3',
    '--progress-path', str(RUNTIME_ROOT / 'milvus-progress.json'),
]
for video in PROCESS_VIDEOS:
    ingest.extend(['--video-id', video.stem])
print(subprocess.check_output(ingest, env=runtime_env, text=True).strip())


In [ ]:
def stop_owned_process(pid_path: Path, required_tokens: tuple[str, ...]) -> None:
    if not pid_path.is_file():
        return
    try:
        pid = int(pid_path.read_text(encoding='utf-8').strip())
        cmdline = Path(f'/proc/{pid}/cmdline').read_bytes().decode('utf-8', errors='replace').replace('\0', ' ')
        if all(token in cmdline for token in required_tokens):
            os.kill(pid, signal.SIGTERM)
    except (FileNotFoundError, ProcessLookupError, PermissionError, ValueError):
        pass

def write_dotenv(path: Path, values: dict[str, str]) -> None:
    temporary = path.with_name('.env.tmp')
    temporary.write_text('\n'.join(f'{key}={json.dumps(value)}' for key, value in values.items()) + '\n', encoding='utf-8')
    temporary.chmod(0o600)
    os.replace(temporary, path)
    path.chmod(0o600)

write_dotenv(BACKEND_ROOT / '.env', {
    'HOST': '127.0.0.1', 'PORT': str(BACKEND_PORT), 'API_PREFIX': '/api',
    'SYSTEM_NAME': 'BoldSearch', 'ZILLIZ_URI': ZILLIZ_URI, 'ZILLIZ_TOKEN': ZILLIZ_TOKEN,
    'MILVUS_COLLECTION': COLLECTION_NAME,
    'MILVUS_OUTPUT_FIELDS': 'frame_id,shot_id,video_id,thumbnail,corpus_version',
    'MILVUS_RANKER_WEIGHTS': '1.0', 'SEARCH_TOP_K': str(SEARCH_TOP_K),
    'LOAD_FG_CLIP_ON_STARTUP': 'true', 'FG_CLIP_DEVICE': 'cuda', 'HF_TOKEN': HF_TOKEN,
    'FRAME_IMAGE_URL_TEMPLATE': '/keyframes/{video_id}/{frame_id}.webp',
    'INCLUDE_EMBEDDING_IN_RESPONSE': 'false',
})
backend_log = RUNTIME_ROOT / 'backend.log'
backend_pid = RUNTIME_ROOT / 'backend.pid'
stop_owned_process(backend_pid, ('boldsearch_integration.fastapi_launcher',))
with backend_log.open('wb') as handle:
    backend_process = subprocess.Popen([
        'uv', 'run', '--frozen', 'python', '-m', 'boldsearch_integration.fastapi_launcher',
        '--app-root', str(BACKEND_ROOT), '--host', '127.0.0.1', '--port', str(BACKEND_PORT),
    ], cwd=BACKEND_ROOT, env=runtime_env, stdin=subprocess.DEVNULL, stdout=handle, stderr=subprocess.STDOUT, start_new_session=True)
backend_pid.write_text(str(backend_process.pid), encoding='utf-8')
from urllib.request import urlopen
health_url = f'http://127.0.0.1:{BACKEND_PORT}/api/health'
deadline = time.monotonic() + 900
while time.monotonic() < deadline:
    if backend_process.poll() is not None:
        break
    try:
        with urlopen(health_url, timeout=5) as response:
            if response.status == 200:
                print('Backend ready:', response.read().decode('utf-8'))
                break
    except Exception:
        time.sleep(2)
else:
    backend_process.terminate()
    raise RuntimeError('Backend did not become ready within 15 minutes.')
if backend_process.poll() is not None:
    raise RuntimeError(backend_log.read_text(encoding='utf-8', errors='replace')[-6000:])


In [ ]:
gateway_log = RUNTIME_ROOT / 'gateway.log'
gateway_pid = RUNTIME_ROOT / 'gateway.pid'
stop_owned_process(gateway_pid, ('boldsearch_integration.gateway',))
gateway_env = runtime_env | {
    'BOLDSEARCH_PUBLIC_ROOT': str(PUBLIC_ROOT),
    'BOLDSEARCH_FRONTEND_DIST': str(FRONTEND_DIST),
    'BOLDSEARCH_BACKEND': f'http://127.0.0.1:{BACKEND_PORT}',
    'BOLDSEARCH_GATEWAY_HOST': '127.0.0.1',
    'BOLDSEARCH_GATEWAY_PORT': str(GATEWAY_PORT),
}
with gateway_log.open('wb') as handle:
    gateway_process = subprocess.Popen([str(BACKEND_PYTHON), '-m', 'boldsearch_integration.gateway'], env=gateway_env, stdin=subprocess.DEVNULL, stdout=handle, stderr=subprocess.STDOUT, start_new_session=True)
gateway_pid.write_text(str(gateway_process.pid), encoding='utf-8')
gateway_health = f'http://127.0.0.1:{GATEWAY_PORT}/api/health'
for _ in range(60):
    try:
        with urlopen(gateway_health, timeout=5) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(gateway_log.read_text(encoding='utf-8', errors='replace')[-6000:])

import base64
import csv
from urllib.request import Request
with (ACTIVE_RELEASE / 'Frames.csv').open(encoding='utf-8', newline='') as handle:
    sample = next(csv.DictReader(handle))
sample_url = f"http://127.0.0.1:{GATEWAY_PORT}/keyframes/{sample['video_id']}/{sample['frame_id']}.webp"
with urlopen(sample_url, timeout=20) as response:
    sample_image = response.read()
    if response.headers.get_content_type() != 'image/webp':
        raise RuntimeError('Gateway did not serve WebP.')
payload = json.dumps({'task': 'VKIS', 'imageCue': {'dataUrl': 'data:image/webp;base64,' + base64.b64encode(sample_image).decode('ascii')}, 'topK': 5}).encode('utf-8')
search_request = Request(f'http://127.0.0.1:{GATEWAY_PORT}/api/search/visual_query', data=payload, headers={'Content-Type': 'application/json'}, method='POST')
with urlopen(search_request, timeout=900) as response:
    search_result = json.loads(response.read().decode('utf-8'))
if not search_result.get('results'):
    raise RuntimeError('Visual search returned no results after ingestion.')

sys.path.insert(0, str(RUNTIME_REPO_ROOT))
from boldsearch_integration.tunnel import ensure_cloudflared, start_quick_tunnel
cloudflared = ensure_cloudflared(Path('/kaggle/working/cloudflared'))
tunnel_process, public_url = start_quick_tunnel(
    cloudflared, f'http://127.0.0.1:{GATEWAY_PORT}',
    log_path=RUNTIME_ROOT / 'tunnel.log', pid_path=RUNTIME_ROOT / 'tunnel.pid',
)
for _ in range(30):
    try:
        with urlopen(public_url + '/api/health', timeout=15) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Quick Tunnel URL was created but public health check failed.')
print('Open BoldSearch:', public_url)
print('Release:', active_release_id, '| keyframes:', manifest['row_count'], '| sample:', sample_url)
